## EDA - MAX

### Objectif

La mission est de construire un modèle de classification binaire pour détecter les cas à risque de chargeback (fraude/abus). Les données sont réalistes, volumineuses (200k lignes) et volontairement imparfaites : valeurs manquantes, outliers, doublons, catégories incohérentes, variables redondantes… L’objectif est de reproduire un vrai contexte data : EDA → nettoyage → feature engineering → modélisation → tuning → choix de seuil.

### Information DataSet

In [2]:
import pandas as pd

df_train = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')
df_train.head()

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,0.39006,0.10963,0.55097,-0.56104,1,NaN,approve,0,7.9,NaN
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,0.03265,-0.40256,0.36218,0.86583,1,NaN,approve,0,5.5,NaN
2,CUST_8ZI3LCBZ0W,ACC_AF53381QSC0L,27,0,20326.87,720.0,25,73.61,492.57,4.6,...,-0.15637,0.57818,0.28902,-2.19864,1,NaN,approve,0,7.2,NaN
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703.0,17,47.53,204.18,25.3,...,-1.02145,0.63908,-0.89190,-0.81592,1,NaN,approve,0,4.4,NaN
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,NaN,594.0,13,99.95,734.09,12.8,...,-0.65771,0.08020,0.17606,0.86739,1,NaN,approve,0,4.9,NaN


Info sur lignes et colonnes :

In [3]:
print(f"Nombre de lignes: {len(df_train)}")
print(f"Nombre de colonnes: {df_train.shape[1]}")

Nombre de lignes: 160000
Nombre de colonnes: 56


Info sur la target (target_is_fraud) :

In [7]:
fraud_dist = df_train['target_is_fraud'].value_counts()
fraud_pct = df_train['target_is_fraud'].value_counts(normalize=True) * 100

print(f"Non-fraude: {fraud_dist[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"Fraude: {fraud_dist[1]:,} ({fraud_pct[1]:.2f}%)")

Non-fraude: 155,076 (96.92%)
Fraude: 4,924 (3.08%)


On observe que le ratio fraude/non-fraude est de 1 pour 31.

On ne peut pas faire confiance à l'accuracy car elle peut être trompeuse

On observe maintenant les valeurs manquantes : 

In [10]:
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train)) * 100
missing_df = pd.DataFrame({
    'Colonne': missing.index,
    'Nb_Manquants': missing.values,
    'Pourcentage': missing_pct.values
}).sort_values('Nb_Manquants', ascending=False)
missing_df = missing_df[missing_df['Nb_Manquants'] > 0]

print(f"Colonnes avec valeurs manquantes: {len(missing_df)}/{df_train.shape[1]}")
print(f"\nTop 10 colonnes avec le plus de valeurs manquantes:")
print(missing_df.head(10).to_string(index=False))

Colonnes avec valeurs manquantes: 14/56

Top 10 colonnes avec le plus de valeurs manquantes:
               Colonne  Nb_Manquants  Pourcentage
partner_risk_indicator        155249    97.030625
  legacy_partner_score        153736    96.085000
       secondary_email        147391    92.119375
                region         45240    28.275000
     annual_income_eur         11186     6.991250
    avg_amount_30d_eur          9547     5.966875
          credit_score          7952     4.970000
    max_amount_30d_eur          7923     4.951875
        device_trust_z          6384     3.990000
         customer_note          4802     3.001250


Corrélation avec Target_is_fraud :

In [12]:
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'target_is_fraud' in numeric_cols:
    numeric_cols.remove('target_is_fraud')

correlations = []
for col in numeric_cols:
    if df_train[col].notna().sum() > 0:
        corr = df_train[[col, 'target_is_fraud']].corr().iloc[0, 1]
        correlations.append({'Feature': col, 'Correlation': corr})

corr_df = pd.DataFrame(correlations).dropna()
corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
corr_df = corr_df.sort_values('Abs_Correlation', ascending=False)

print(f"\nTop 15 features les plus corrélées avec target_is_fraud:")
print(corr_df.head(15)[['Feature', 'Correlation']].to_string(index=False))


Top 15 features les plus corrélées avec target_is_fraud:
                        Feature  Correlation
chargeback_resolution_time_days     0.674610
         post_event_status_code     0.639533
                num_devices_30d     0.066151
                      ip_risk_z     0.062770
        tx_amount_total_30d_eur     0.048169
             avg_amount_30d_eur     0.046508
                 device_trust_z    -0.042938
                   credit_score    -0.035127
              credit_score_norm    -0.034618
             max_amount_30d_eur     0.034276
                         is_vpn     0.032702
                  is_new_device     0.028932
                     income_log    -0.028129
                  tenure_months    -0.026540
        income_estimate_alt_eur    -0.024909


chargeback_resolution_time_days     0.674610

post_event_status_code     0.639533

C'est deux colonnes sont très corrélés avec la cible. Or, ici ces valeurs ne sont connus que après et dans un cas concret, nous n'aurons pas ces informations. Donc elles sont à supprimer

In [14]:
cat_cols = df_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
bool_cols = df_train.select_dtypes(include=['bool']).columns.tolist()

print(f"Colonnes numériques: {len(num_cols)}")
print(f"Colonnes catégorielles: {len(cat_cols)}")
print(f"Colonnes booléennes: {len(bool_cols)}")

Colonnes numériques: 36
Colonnes catégorielles: 20
Colonnes booléennes: 0
